<a href="https://colab.research.google.com/github/1umesh/gpt-model/blob/main/gpt_scrap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


--2026-02-27 11:45:22--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-02-27 11:45:23 (31.2 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print(len(text))

1115394


In [4]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [5]:
print(set(text))

{'T', 'c', "'", 'L', 'a', 'j', 'w', 'd', 'S', 'G', '&', 'f', 'u', 'o', 'b', 't', '.', 'x', 'v', '\n', 'y', ':', ' ', 'B', 'n', '-', 'A', 'p', 'C', 'r', 'H', 'F', 'E', '3', 'g', 'P', 'K', 's', 'k', 'm', 'l', 'h', 'R', 'J', 'Y', '!', 'Z', 'U', 'z', 'i', 'I', 'e', ',', ';', 'q', 'O', 'N', '$', '?', 'M', 'X', 'D', 'W', 'Q', 'V'}


In [6]:
# intro on how set will work here
s="hello world hello python"
words=s.split()
print(words)
print(type(words))
print(set(words))

['hello', 'world', 'hello', 'python']
<class 'list'>
{'hello', 'world', 'python'}


In [7]:
# all the character appered in our data(text file)
char=sorted(list(set(text)))
vocab_size=len(char)
print("".join(char))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [8]:
# encoder and decoder for our tokeniser
#here we are mapping our char to unique numbers which represent them . kind of like vector spacing
stoi={ch:i for i,ch in enumerate(char)} #here character is key and it's index is value
itoi={i:ch for i,ch in enumerate (char)} # her opposite of above
encoder=lambda s:[stoi[c] for c in s] # function to encode the string

decoder =lambda i:"".join(itoi[c] for c in i) # funcrtion to decode the numerical array(encoded values)

In [9]:
print(encoder("jelly fish"))
print(decoder([48, 43, 50, 50, 63, 1, 44, 47, 57, 46]))

[48, 43, 50, 50, 63, 1, 44, 47, 57, 46]
jelly fish


In [10]:
import torch
data=torch.tensor(encoder(text),dtype=torch.long)
print(data.shape,data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [11]:
# splitting our data for testing and training
n=int(0.8*len(data))
train=data[:n]
test=data[n:]

In [12]:
print(torch.randint(10,(9,2)))

tensor([[4, 5],
        [6, 3],
        [3, 7],
        [5, 1],
        [2, 0],
        [4, 2],
        [6, 4],
        [6, 8],
        [2, 0]])


In [13]:
torch.manual_seed(1)
batch_size=4
block_size=8

def get_batch(split):
  data=train if split=="train" else test
  start_x=torch.randint(len(data)-block_size,(batch_size,))
  x=torch.stack([data[i:i+block_size]for i in start_x])
  y=torch.stack([data[i+1:i+block_size+1]for i in start_x])
  return x,y
xb,yb=get_batch("train")
print("inputs")
print(xb.shape)
print(xb.dtype)
print(xb)
print(xb.view(4*8))

inputs
torch.Size([4, 8])
torch.int64
tensor([[39, 51, 12,  0,  0, 31, 32, 13],
        [43, 43, 58,  1, 58, 46, 43,  1],
        [43,  1, 57, 58, 53, 41, 49, 57],
        [40, 56, 43, 39, 57, 58,  1, 57]])
tensor([39, 51, 12,  0,  0, 31, 32, 13, 43, 43, 58,  1, 58, 46, 43,  1, 43,  1,
        57, 58, 53, 41, 49, 57, 40, 56, 43, 39, 57, 58,  1, 57])


In [14]:
import torch
import torch.nn as nn

embedding = nn.Embedding(10, 4)  # 10 words, each represented by 4 numbers

input = torch.tensor([[1, 3, 5],[2,4,6]])
output = embedding(input)

print(output)
print(output.shape)#(batch_size,sequence_lenght,embedding_dim)
print(" 2 sentence -> 3 words each -> 4 feature per words")

tensor([[[-1.1660, -1.0748,  1.3168, -0.6818],
         [-0.5710,  0.0135, -0.5495, -1.0113],
         [ 0.4391,  1.1712,  1.7674, -0.0954]],

        [[-0.2223,  1.6871,  0.2284,  0.4676],
         [ 0.8657,  0.2444, -0.6629,  0.8073],
         [ 0.0981, -1.2150,  0.7312,  1.1718]]], grad_fn=<EmbeddingBackward0>)
torch.Size([2, 3, 4])
 2 sentence -> 3 words each -> 4 feature per words


In [16]:
import torch.nn as nn
from torch.nn import functional as f
torch.manual_seed(1)
# we first convert string into intergers using encoder we made earlier
# now we can't those list of integer directly to model bcuse it might make some
#other meanin from it . like feeding[0,1,2] , model might thinks 2>1>0.
#these are just ids/indexes not real values. so we use nn.Embedding to map each number to a vector


class bigramlm(nn.Module):
  def __init__ (self,vocab_size):
    super().__init__() # it is used to call constructor of base/parent class .
    # in this case constructor of nn.Module
    #nn.Embedding(num_embeddings,embedding_dim)
    self.token_embedding_table=nn.Embedding(vocab_size,vocab_size)
    # self.llm_head=nn.Linear(n_embd,vocab_size) this will give probability of every word/vocab
    # and best one is our predicted output

  def forward(self,index,target=None):
    logits=self.token_embedding_table(index)
    if target==None:
      loss=None
    else:
      b,t,c=logits.shape #(batch_size,sequence_lenght,embedding_dim)
      logits=logits.view(b*t,c)# view () is used to reshape the tensor
      target=target.view(b*t)
      loss=f.cross_entropy(logits,target)
    return logits,loss
m=bigramlm(vocab_size)
logits,loss=m(xb,yb)
print(logits.shape)
print(loss)

torch.Size([32, 65])
tensor(4.6057, grad_fn=<NllLossBackward0>)


In [23]:
# self attention
torch.manual_seed(1)
b,t,c=4,8,32 #batch,time,channel
x=torch.randn(b,t,c)

#single head self attention
head_size=16
key=nn.Linear(c,head_size,bias=False)
query=nn.Linear(c,head_size,bias=False)
value=nn.Linear(c,head_size,bias=False)

k=key(x) #b,t,16
q=query(x)
v=value(x)

weight=q@k.transpose(-2,-1)*(head_size**(-0.5))
#it is divided by sqrt of head size to reduce the variance of weight
#if we don't do it then variance of weight come close to head size
#b,t,16 @ b,16,t -> b,t,t
lower_tri= torch.tril(torch.ones(t,t))
weight=weight.masked_fill(lower_tri==0,float('-inf')) # where lower_tri==0 is  change it with float("-inf")
weight=f.softmax(weight,dim=-1)

out=weight@v

out.shape

torch.Size([4, 8, 16])

In [24]:
out[0]

tensor([[-0.4672, -0.3683, -0.0126, -0.0979, -0.2627, -0.2885, -0.2783,  0.2652,
          0.3973, -0.6045, -0.4596, -0.0694,  1.1627, -0.5240,  0.1212,  0.0297],
        [-0.4482, -0.2981,  0.1077, -0.0241,  0.1440,  0.1002, -0.1000, -0.0138,
          0.0982, -0.2496, -0.1516, -0.1244,  0.9826, -0.5019,  0.4260, -0.0180],
        [-0.2758, -0.2726,  0.0718, -0.0682,  0.0389,  0.1411, -0.0438, -0.4461,
         -0.0591, -0.1077, -0.0055, -0.5231,  0.6717,  0.1290,  0.4222,  0.3048],
        [-0.1139, -0.4460, -0.0412, -0.3906, -0.3237,  0.1150, -0.1679, -0.2663,
          0.0849, -0.2569, -0.0404, -0.3629,  0.4762,  0.2137,  0.0922,  0.1865],
        [ 0.1476, -0.7730, -0.0818, -0.3200, -0.4005, -0.1137, -0.1591, -0.7982,
          0.1872, -0.0913, -0.0097, -0.2684,  0.1096,  0.2746, -0.2170,  0.3779],
        [ 0.1478, -0.7037, -0.1282, -0.3869, -0.3795, -0.0925, -0.1435, -0.7871,
          0.1344, -0.1340,  0.0834, -0.3410,  0.0744,  0.3268, -0.1842,  0.3369],
        [ 0.1375, -0.4

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))
